# Deep Learning Training Dynamics: Normalization, Schedules & Regularization

This interactive lab covers:
1. **Normalization Mechanics**: Comparing Batch Normalization (across samples) vs. Layer Normalization (across features).
2. **Learning Rate Schedules**: Implementing and visualizing Cosine Annealing with Linear Warmup.
3. **Gradient Norm Clipping**: Taming exploding gradients in deep and recurrent architectures.
4. **Early Stopping Callback**: Preventing overfitting and automating model checkpointing.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / 'code'))

import math
import torch
import torch.nn as nn
from training_toolkit import (
    ScratchBatchNorm1d, ScratchLayerNorm,
    get_cosine_warmup_lr, clip_grad_norm_scratch,
    EarlyStoppingHandler
)

print('Training Toolkit loaded successfully!')

## 1. Normalization: BatchNorm vs. LayerNorm

We pass an unnormalized feature matrix through `ScratchBatchNorm1d` and `ScratchLayerNorm` to examine normalization axes.

In [ ]:
torch.manual_seed(42)
# Raw inputs with mean ~10.0 and variance ~25.0
x = torch.randn(8, 4) * 5.0 + 10.0

bn = ScratchBatchNorm1d(num_features=4)
ln = ScratchLayerNorm(normalized_shape=4)

bn.train()
out_bn = bn(x)
out_ln = ln(x)

print('Raw Input Mean (per feature):', x.mean(dim=0).numpy().round(2))
print('BatchNorm Output Mean (per feature):', out_bn.mean(dim=0).detach().numpy().round(4))
print('LayerNorm Output Mean (per sample):', out_ln.mean(dim=-1).detach().numpy().round(4))

## 2. Learning Rate Schedule: Cosine Annealing with Warmup

We compute the schedule trajectory over 100 optimization steps with a 15-step linear warmup.

In [ ]:
total_steps = 100
warmup_steps = 15
base_lr = 1e-3
min_lr = 1e-5

steps = [0, 5, 15, 30, 50, 75, 99]
print(f'{"Step":<8} | {"Phase":<16} | {"Learning Rate":<16}')
print('-' * 44)
for s in steps:
    lr = get_cosine_warmup_lr(s, total_steps, warmup_steps, base_lr, min_lr)
    phase = 'Linear Warmup' if s < warmup_steps else 'Cosine Decay'
    print(f'{s:<8} | {phase:<16} | {lr:<16.6f}')

## 3. Gradient Norm Clipping Demonstration

We simulate an exploding gradient update ($\|g\|_2 = 25.0$) and rescale it to a maximum threshold of $1.0$.

In [ ]:
param = nn.Parameter(torch.ones(5))
param.grad = torch.ones(5) * 11.18  # Norm = sqrt(5 * 11.18^2) ~ 25.0

orig_norm = math.sqrt(param.grad.pow(2).sum().item())
max_norm = 1.0
clip_grad_norm_scratch([param], max_norm=max_norm)
clipped_norm = math.sqrt(param.grad.pow(2).sum().item())

print(f'Original Gradient Norm: {orig_norm:.2f}')
print(f'Clipped  Gradient Norm: {clipped_norm:.2f}')
assert math.isclose(clipped_norm, max_norm, rel_tol=1e-3)

## 4. Automated Early Stopping Callback

We simulate 10 epochs of validation losses and observe early termination when loss stops improving.

In [ ]:
handler = EarlyStoppingHandler(patience=3, min_delta=0.005)
simulated_val_losses = [1.20, 1.05, 0.90, 0.82, 0.819, 0.821, 0.820, 0.822]

for epoch, v_loss in enumerate(simulated_val_losses):
    stop_flag = handler.step(v_loss)
    status = f'Counter: {handler.counter}/{handler.patience}' if not stop_flag else 'STOPPING TRIGGERED'
    print(f'Epoch {epoch + 1}: Val Loss = {v_loss:.3f} | Best = {handler.best_loss:.3f} | {status}')
    if stop_flag:
        break